[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/LegalIntermediaSL/Physics/blob/main/Simulaciones_Jupyter/Mecanica_Cuantica/Ecuacion_Schrodinger_Simulacion.ipynb)



# Simulación Computacional: Ecuacion Schrodinger
**Área:** Mecanica Cuantica

Este cuaderno interactivo de Jupyter ejecuta numéricamente los modelos físicos descritos en el repositorio. Puedes modificar los parámetros iniciales y ejecutar las celdas para visualizar gráficos o animaciones interactivos.

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve

# Parámetros físicos
L = 10.0         # Longitud del dominio
N = 500          # Número de puntos espaciales
x = np.linspace(0, L, N)
dx = x[1] - x[0]
dt = 0.005       # Paso de tiempo
steps = 200      # Número de pasos de tiempo a simular

# Potencial (pozo armónico)
V = 0.5 * (x - L/2)**2
V[0] = V[-1] = 1e6 # Condiciones de frontera infinitas

# Paquete de ondas inicial (Gaussiana)
x0 = L / 4.0
sigma = 0.5
k0 = 5.0
psi0 = np.exp(-0.5 * ((x - x0) / sigma)**2) * np.exp(1j * k0 * x)
psi0 = psi0 / np.sqrt(np.sum(np.abs(psi0)**2) * dx) # Normalización

# Operador de evolución (Crank-Nicolson)
alpha = 1j * dt / (4 * dx**2)
main_diag_A = 1 + 2*alpha + 1j * dt / 2 * V
off_diag_A = -alpha * np.ones(N-1)
main_diag_B = 1 - 2*alpha - 1j * dt / 2 * V
off_diag_B = alpha * np.ones(N-1)

A = diags([off_diag_A, main_diag_A, off_diag_A], [-1, 0, 1], format='csc')
B = diags([off_diag_B, main_diag_B, off_diag_B], [-1, 0, 1], format='csc')

# Simulación
psi = psi0.copy()
densities = []

for _ in range(steps):
    densities.append(np.abs(psi)**2)
    # Resolver A * psi_new = B * psi_old
    rhs = B.dot(psi)
    psi = spsolve(A, rhs)

# Visualización
fig, ax = plt.subplots(figsize=(8, 5))
line, = ax.plot(x, densities[0], color='blue', lw=2, label='|Ψ|²')
ax.plot(x, V * 0.1, color='red', linestyle='--', label='Potencial V(x) (escalado)')
ax.set_ylim(0, np.max(densities) * 1.2)
ax.set_xlim(0, L)
ax.set_xlabel('Posición x')
ax.set_ylabel('Densidad de Probabilidad')
ax.set_title('Evolución de un Paquete de Ondas Cuántico')
ax.legend(loc='upper right')

def animate(i):
    line.set_ydata(densities[i])
    return line,

ani = animation.FuncAnimation(fig, animate, frames=steps, interval=50, blit=True)
plt.tight_layout()
# plt.show() # Descomentar para ver la animación localmente